<a href="https://colab.research.google.com/github/radsrei/Caderno-Faculdade/blob/main/fuzzy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# --- PARTE 1: PÊNDULO (Ângulo e Velocidade Angular) ---

# Vértices das FPs do Ângulo (theta)
pendulo_theta = {
    'N': [-0.15, -0.15, -0.1, 0],
    'Z': [-0.1, -0.03, 0.03, 0.1],
    'P': [0, 0.1, 0.15, 0.15]
}

# Vértices das FPs da Velocidade Angular (theta_dot) [cite: 83]
pendulo_v_angular = {
    'N': [-0.15, -0.15, -0.1, 0],
    'Z': [-0.15, -0.03, 0.03, 0.15], # Ajustado conforme tabela pág 3
    'P': [0, 0.1, 0.15, 0.15]
}

# Matriz de Regras do Pêndulo [cite: 43-51]
# Linhas: Velocidade (N, Z, P) | Colunas: Ângulo (N, Z, P)
regras_pendulo = [
    ['NL', 'NS', 'Z'],  # Velocidade N
    ['NM', 'Z', 'PM'],  # Velocidade Z
    ['Z', 'PS', 'PL']   # Velocidade P
]

# --- PARTE 2: CARRO (Posição e Velocidade Linear) ---

# Vértices das FPs da Posição (x) [cite: 156-171]
carro_posicao = {
    'N': [-3, -3, -1.5, 0],
    'Z': [-1.5, -0.5, 0.5, 1.5],
    'P': [0, 1.5, 3, 3]
}

# Vértices das FPs da Velocidade do Carro (x_dot) [cite: 187]
carro_v_linear = {
    'N': [-3, -3, -1.5, 0],
    'Z': [-1.5, -0.5, 0.5, 1.5], # Sugerido pela simetria do gráfico pág 6
    'P': [0, 1.5, 3, 3]
}

# Matriz de Regras do Carro [cite: 124-134]
regras_carro = [
    ['PL', 'PS', 'Z'],  # Velocidade N
    ['PM', 'Z', 'NM'],  # Velocidade Z
    ['Z', 'NS', 'NL']   # Velocidade P
]

# --- SAÍDAS (Força F) ---

# FPs Saída Pêndulo (Triangulares) [cite: 98]
saida_forca_pendulo = {
    'NL': [-200, -100, 0], 'NM': [-80, -40, 0], 'NS': [-10, -5, 0],
    'Z': [0, 0, 0],
    'PS': [0, 5, 10], 'PM': [0, 40, 80], 'PL': [0, 100, 200]
}

# FPs Saída Carro (Triangulares) [cite: 201]
saida_forca_carro = {
    'NL': [-100, -50, 0], 'NM': [-10, -5, 0], 'NS': [-2, -1, 0],
    'Z': [0, 0, 0],
    'PS': [0, 1, 2], 'PM': [0, 5, 10], 'PL': [0, 50, 100]
}

In [2]:
def trimf(x, params):
    """Função de Pertinência Triangular"""
    a, b, c = params
    return max(0, min((x - a) / (b - a) if b > a else 1, (c - x) / (c - b) if c > b else 1))

def trapmf(x, params):
    """Função de Pertinência Trapezoidal"""
    a, b, c, d = params
    term1 = (x - a) / (b - a) if b > a else 1
    term2 = (d - x) / (d - c) if d > c else 1
    return max(0, min(term1, 1, term2))

def defuzz_weighted_average(forcas, centros):
    """Cálculo de Defuzzificação por Média Ponderada [cite: 205]"""
    numerador = sum(f * c for f, c in zip(forcas, centros))
    denominador = sum(forcas)
    return numerador / denominador if denominador != 0 else 0

In [7]:
def main():
    # Parâmetros de Simulação [cite: 215-216, 225]
    mc, mp, l, g, I = 0.5, 0.2, 0.3, 9.8, 0.006
    h = 0.02  # Passo de tempo

    # Estado Inicial: [x, x_dot, theta, theta_dot]
    # Exemplo: Pêndulo levemente inclinado (0.1 rad)
    state = [0.0, 0.0, 0.1, 0.0]

    print(f"{'Tempo':>6} | {'Posição':>8} | {'Ângulo':>8} | {'Força':>8}")
    print("-" * 45)

    for step in range(500):
        x, x_dot, theta, theta_dot = state

        # 1. FUZZIFICAÇÃO E INFERÊNCIA (Loop de Controle) [cite: 226-228]

        # --- Parte 1: Pêndulo ---
        mu_theta = {k: trapmf(theta, v) for k, v in pendulo_theta.items()}
        mu_v_ang = {k: trapmf(theta_dot, v) for k, v in pendulo_v_angular.items()}

        forcas_p, pesos_p = [], []
        # Avaliação das 9 regras do pêndulo [cite: 34-42, 47-51]
        for i, label_v in enumerate(['N', 'Z', 'P']):
            for j, label_t in enumerate(['N', 'Z', 'P']):
                peso = min(mu_v_ang[label_v], mu_theta[label_t])
                if peso > 0:
                    consequente = regras_pendulo[i][j]
                    centro = saida_forca_pendulo[consequente][1] # Vértice B
                    forcas_p.append(centro)
                    pesos_p.append(peso)

        f_pendulo = defuzz_weighted_average(pesos_p, forcas_p)

        # --- Parte 2: Carro ---
        mu_pos = {k: trapmf(x, v) for k, v in carro_posicao.items()}
        mu_v_lin = {k: trapmf(x_dot, v) for k, v in carro_v_linear.items()}

        forcas_c, pesos_c = [], []
        # Avaliação das 9 regras do carro [cite: 114-123, 129-134]
        for i, label_v in enumerate(['N', 'Z', 'P']):
            for j, label_p in enumerate(['N', 'Z', 'P']):
                peso = min(mu_v_lin[label_v], mu_pos[label_p])
                if peso > 0:
                    consequente = regras_carro[i][j]
                    centro = saida_forca_carro[consequente][1]
                    forcas_c.append(centro)
                    pesos_c.append(peso)

        f_carro = defuzz_weighted_average(pesos_c, forcas_c)

        # Força Total Aplicada [cite: 216, 227]
        F = f_pendulo + f_carro

        # 2. DINÂMICA DO SISTEMA (Atualização do Estado) [cite: 208, 220-224]
        # Cálculo das acelerações
        sin_t, cos_t = np.sin(theta), np.cos(theta)

        # Aceleração linear (x_double_dot)
        x_acc = (mp * l * (theta_dot**2 * sin_t) + F) / (mc + mp) # Simplificada conforme pág 7

        # Aceleração angular (theta_double_dot)
        theta_acc = (mp * l * (g * sin_t - x_acc * cos_t)) / (I + mp * l**2)

        # Integração de Euler [cite: 221-224]
        state[1] += h * x_acc      # x_dot
        state[0] += h * state[1]   # x
        state[3] += h * theta_acc  # theta_dot
        state[2] += h * state[3]   # theta

        if step % 10 == 0:
            print(f"{step*h:6.2f}s | {state[0]:8.4f} | {state[2]:8.4f} | {F:8.2f}N")

if __name__ == "__main__":
    main()

 Tempo |  Posição |   Ângulo |    Força
---------------------------------------------
  0.00s |   0.0229 |   0.0441 |    40.00N
  0.20s |   0.0464 |  -0.0037 |     8.43N
  0.40s |   0.0487 |   0.0011 |    11.32N
  0.60s |   0.0535 |   0.0004 |     9.08N
  0.80s |   0.0590 |  -0.0007 |     7.46N
  1.00s |   0.0652 |  -0.0026 |     6.66N
  1.20s |   0.0717 |  -0.0039 |     7.27N
  1.40s |   0.0782 |  -0.0038 |     9.38N
  1.60s |   0.0847 |  -0.0021 |    11.10N
  1.80s |   0.0914 |   0.0006 |    11.36N
  2.00s |   0.0999 |   0.0008 |     9.83N
  2.20s |   0.1105 |  -0.0023 |     7.08N
  2.40s |   0.1212 |  -0.0036 |     9.40N
  2.60s |   0.1315 |  -0.0016 |    11.89N
  2.80s |   0.1420 |   0.0022 |    11.97N
  3.00s |   0.1574 |  -0.0032 |     6.56N
  3.20s |   0.1713 |  -0.0016 |    14.33N
  3.40s |   0.1850 |   0.0044 |    13.59N
  3.60s |   0.2042 |   0.0001 |    10.77N
  3.80s |   0.2235 |  -0.0010 |    10.13N
  4.00s |   0.2434 |  -0.0000 |    12.97N
  4.20s |   0.2641 |   0.0032 | 

##Revisado pelo Claudio

In [10]:
"""
Controlador Fuzzy: Pêndulo Invertido sobre Carro
=================================================
Dois controladores fuzzy em cascata:
  1. Controlador do Pêndulo  → estabiliza o ângulo theta
  2. Controlador do Carro    → regula a posição x

Defuzzificação: Média Ponderada (Weighted Average)
Integração:     Euler Semi-Implícito (Symplectic Euler)
Dinâmica:       Equações completas do pêndulo-carro acoplado
"""

from __future__ import annotations

import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple


# ---------------------------------------------------------------------------
# Estruturas de dados
# ---------------------------------------------------------------------------

@dataclass
class FuzzySystem:
    """Agrupa as FPs de entrada, regras e FPs de saída de um controlador."""
    var1_mfs: Dict[str, List[float]]   # primeira variável de entrada (e.g. ângulo)
    var2_mfs: Dict[str, List[float]]   # segunda variável de entrada (e.g. vel. angular)
    rules: List[List[str]]             # matriz 3x3 de consequentes
    output_mfs: Dict[str, List[float]] # FPs de saída (triangulares: [a, b, c])
    labels: List[str] = field(default_factory=lambda: ['N', 'Z', 'P'])


# ---------------------------------------------------------------------------
# Funções de pertinência
# ---------------------------------------------------------------------------

def trapmf(x: float, params: List[float]) -> float:
    """Função de pertinência trapezoidal."""
    a, b, c, d = params
    left  = (x - a) / (b - a) if b > a else 1.0
    right = (d - x) / (d - c) if d > c else 1.0
    return float(np.clip(min(left, right), 0.0, 1.0))


def fuzzify(x: float, mfs: Dict[str, List[float]]) -> Dict[str, float]:
    """Calcula o grau de pertinência de x em cada conjunto fuzzy."""
    return {label: trapmf(x, params) for label, params in mfs.items()}


# ---------------------------------------------------------------------------
# Motor de inferência + defuzzificação
# ---------------------------------------------------------------------------

def infer_and_defuzz(
    mu1: Dict[str, float],
    mu2: Dict[str, float],
    rules: List[List[str]],
    output_mfs: Dict[str, List[float]],
    labels: List[str],
) -> float:
    """
    Inferência Mamdani (t-norma = mínimo) com defuzzificação
    por Média Ponderada usando o vértice central (b) das FPs triangulares.

    Retorna a força defuzzificada (float).
    """
    weighted_sum = 0.0
    weight_total = 0.0

    for i, lv1 in enumerate(labels):
        for j, lv2 in enumerate(labels):
            firing = min(mu1[lv1], mu2[lv2])   # grau de ativação da regra
            if firing <= 0.0:
                continue
            consequent = rules[i][j]
            center = output_mfs[consequent][1]  # vértice b da triangular
            weighted_sum += firing * center
            weight_total += firing

    return weighted_sum / weight_total if weight_total > 0.0 else 0.0


def compute_control_force(system: FuzzySystem, val1: float, val2: float) -> float:
    """Interface de alto nível: fuzzifica → infere → defuzzifica."""
    mu1 = fuzzify(val1, system.var1_mfs)
    mu2 = fuzzify(val2, system.var2_mfs)
    return infer_and_defuzz(mu1, mu2, system.rules, system.output_mfs, system.labels)


# ---------------------------------------------------------------------------
# Dinâmica do sistema pêndulo-carro
# ---------------------------------------------------------------------------

@dataclass
class CartPoleParams:
    mc: float = 0.5    # massa do carro [kg]
    mp: float = 0.2    # massa do pêndulo [kg]
    l:  float = 0.3    # comprimento da haste [m]
    g:  float = 9.8    # aceleração gravitacional [m/s²]
    I:  float = 0.006  # momento de inércia [kg·m²]


def equations_of_motion(
    state: np.ndarray,
    F: float,
    p: CartPoleParams,
) -> Tuple[float, float]:
    """
    Equações completas do pêndulo-carro acoplado.

    Derivadas de Lagrange:
      (mc + mp)*x'' + mp*l*(theta''*cos - theta'^2*sin) = F
      (I + mp*l²)*theta'' + mp*l*x''*cos               = mp*g*l*sin

    Retorna (x_acc, theta_acc).
    """
    _, x_dot, theta, theta_dot = state
    sin_t, cos_t = np.sin(theta), np.cos(theta)

    Il  = p.I + p.mp * p.l ** 2
    M   = p.mc + p.mp
    ml  = p.mp * p.l

    # Matriz 2×2 e vetor de forças generalizadas
    # [ M           ml*cos ] [x'']     = [ F + ml*theta'^2*sin ]
    # [ ml*cos      Il     ] [theta''] = [ mp*g*l*sin          ]

    det = M * Il - (ml * cos_t) ** 2

    rhs_x     = F + ml * theta_dot ** 2 * sin_t
    rhs_theta = p.mp * p.g * p.l * sin_t

    x_acc     = (Il * rhs_x     - ml * cos_t * rhs_theta) / det
    theta_acc = (M  * rhs_theta - ml * cos_t * rhs_x    ) / det

    return x_acc, theta_acc


def step_euler_symplectic(
    state: np.ndarray,
    F: float,
    p: CartPoleParams,
    h: float,
) -> np.ndarray:
    """
    Integração Euler Semi-Implícito (Symplectic Euler):
      1. Atualiza velocidades com aceleração no instante atual
      2. Atualiza posições com as velocidades *novas*
    Preserva melhor a energia do que o Euler explícito padrão.
    """
    x_acc, theta_acc = equations_of_motion(state, F, p)

    new_state = state.copy()
    new_state[1] += h * x_acc        # x_dot   ← vel. nova
    new_state[3] += h * theta_acc    # theta_dot ← vel. nova
    new_state[0] += h * new_state[1] # x       ← posição com vel. nova
    new_state[2] += h * new_state[3] # theta   ← posição com vel. nova

    return new_state


# ---------------------------------------------------------------------------
# Definição dos controladores fuzzy
# ---------------------------------------------------------------------------

def build_pendulum_controller() -> FuzzySystem:
    theta_mfs = {
        'N': [-0.15, -0.15, -0.10,  0.00],
        'Z': [-0.10, -0.03,  0.03,  0.10],
        'P': [ 0.00,  0.10,  0.15,  0.15],
    }
    omega_mfs = {
        'N': [-0.15, -0.15, -0.10,  0.00],
        'Z': [-0.15, -0.03,  0.03,  0.15],
        'P': [ 0.00,  0.10,  0.15,  0.15],
    }
    rules = [
        ['NL', 'NS', 'Z' ],   # vel. angular N
        ['NM', 'Z',  'PM'],   # vel. angular Z
        ['Z',  'PS', 'PL'],   # vel. angular P
    ]
    output_mfs = {
        'NL': [-200, -100,   0],
        'NM': [ -80,  -40,   0],
        'NS': [ -10,   -5,   0],
        'Z':  [   0,    0,   0],
        'PS': [   0,    5,  10],
        'PM': [   0,   40,  80],
        'PL': [   0,  100, 200],
    }
    return FuzzySystem(theta_mfs, omega_mfs, rules, output_mfs)


def build_cart_controller() -> FuzzySystem:
    pos_mfs = {
        'N': [-3.0, -3.0, -1.5,  0.0],
        'Z': [-1.5, -0.5,  0.5,  1.5],
        'P': [ 0.0,  1.5,  3.0,  3.0],
    }
    vel_mfs = {
        'N': [-3.0, -3.0, -1.5,  0.0],
        'Z': [-1.5, -0.5,  0.5,  1.5],
        'P': [ 0.0,  1.5,  3.0,  3.0],
    }
    rules = [
        ['PL', 'PS', 'Z' ],   # vel. linear N
        ['PM', 'Z',  'NM'],   # vel. linear Z
        ['Z',  'NS', 'NL'],   # vel. linear P
    ]
    output_mfs = {
        'NL': [-100,  -50,   0],
        'NM': [ -10,   -5,   0],
        'NS': [  -2,   -1,   0],
        'Z':  [   0,    0,   0],
        'PS': [   0,    1,   2],
        'PM': [   0,    5,  10],
        'PL': [   0,   50, 100],
    }
    return FuzzySystem(pos_mfs, vel_mfs, rules, output_mfs)


# ---------------------------------------------------------------------------
# Limites de segurança
# ---------------------------------------------------------------------------

THETA_LIMIT = np.pi / 2   # pêndulo caiu se |theta| > 90°
X_LIMIT     = 2.4         # carro saiu dos trilhos se |x| > 2.4 m


def is_terminal(state: np.ndarray) -> bool:
    return abs(state[2]) > THETA_LIMIT or abs(state[0]) > X_LIMIT


# ---------------------------------------------------------------------------
# Simulação principal
# ---------------------------------------------------------------------------

def simulate(
    initial_state: List[float],
    n_steps: int = 500,
    h: float = 0.02,
    print_every: int = 10,
) -> List[np.ndarray]:
    """
    Executa a simulação e retorna a trajetória de estados.

    Parâmetros
    ----------
    initial_state : [x, x_dot, theta, theta_dot]
    n_steps       : número de passos de integração
    h             : passo de tempo [s]
    print_every   : intervalo de impressão (0 = silencioso)
    """
    pendulum_ctrl = build_pendulum_controller()
    cart_ctrl     = build_cart_controller()
    params        = CartPoleParams()

    state = np.array(initial_state, dtype=float)
    trajectory = [state.copy()]

    if print_every > 0:
        header = f"{'Tempo':>6} | {'Posição (m)':>11} | {'Ângulo (rad)':>12} | {'Força (N)':>10}"
        print(header)
        print("-" * len(header))

    for step in range(n_steps):
        x, x_dot, theta, theta_dot = state

        # Controle
        f_pendulo = compute_control_force(pendulum_ctrl, theta,  theta_dot)
        f_carro   = compute_control_force(cart_ctrl,     x,      x_dot    )
        F = f_pendulo + f_carro

        # Integração
        state = step_euler_symplectic(state, F, params, h)
        trajectory.append(state.copy())

        if print_every > 0 and step % print_every == 0:
            t = step * h
            print(f"{t:6.2f}s | {state[0]:11.4f} | {state[2]:12.4f} | {F:10.2f}")

        if is_terminal(state):
            if print_every > 0:
                print(f"\n[!] Sistema divergiu no passo {step} (t={step*h:.2f}s)")
            break

    return trajectory


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Estado inicial: pêndulo levemente inclinado (0.1 rad)
    initial = [0.0, 0.0, 0.1, 0.0]
    trajectory = simulate(initial, n_steps=500, h=0.02, print_every=10)

    final = trajectory[-1]
    print(f"\nEstado final:")
    print(f"  x       = {final[0]:.4f} m")
    print(f"  x_dot   = {final[1]:.4f} m/s")
    print(f"  theta   = {final[2]:.4f} rad")
    print(f"  th_dot  = {final[3]:.4f} rad/s")

 Tempo | Posição (m) | Ângulo (rad) |  Força (N)
------------------------------------------------
  0.00s |      0.0035 |       0.0922 |       5.00
  0.20s |      0.0400 |       0.0330 |     -73.05
  0.40s |      0.0643 |       0.0096 |     -56.46
  0.60s |      0.0677 |       0.0413 |     -53.68
  0.80s |      0.0926 |       0.0242 |     -57.71
  1.00s |      0.1087 |       0.0339 |     -55.76
  1.20s |      0.1329 |       0.0302 |     -57.20
  1.40s |      0.1573 |       0.0332 |     -56.73
  1.60s |      0.1861 |       0.0332 |     -57.57
  1.80s |      0.2183 |       0.0344 |     -57.72
  2.00s |      0.2545 |       0.0354 |     -58.21
  2.20s |      0.2954 |       0.0364 |     -58.63
  2.40s |      0.3409 |       0.0382 |     -59.35
  2.60s |      0.3921 |       0.0402 |     -60.30
  2.80s |      0.4497 |       0.0426 |     -61.44
  3.00s |      0.5145 |       0.0458 |     -63.02
  3.20s |      0.5884 |       0.0486 |     -64.41
  3.40s |      0.6724 |       0.0518 |     -66.05
  

In [11]:
"""
Controlador Fuzzy: Pêndulo Invertido sobre Carro
=================================================
Implementação fiel à especificação do Prof. Claudinei Dias (Ney).

Dois controladores FIS Mamdani em cascata:
  1. Controlador do Pêndulo  → entradas: theta, theta_dot  → saída: F_pêndulo
  2. Controlador do Carro    → entradas: x, x_dot          → saída: F_carro

Defuzzificação : Weighted Average (média ponderada, pág. 7)
Integração     : Euler Explícito (loop da pág. 8)
Dinâmica       : Equações acopladas sequenciais (pág. 7)

Parâmetros físicos (pág. 7):
  mc = 0.5 kg | mp = 0.2 kg | l = 0.3 m | g = 9.8 m/s² | I = 0.006 kg·m²
"""

from __future__ import annotations

import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple


# ---------------------------------------------------------------------------
# Estruturas de dados
# ---------------------------------------------------------------------------

@dataclass
class FuzzySystem:
    """Agrupa FPs de entrada, matriz de regras e FPs de saída de um controlador."""
    var1_mfs: Dict[str, List[float]]    # 1ª entrada (ângulo / posição)
    var2_mfs: Dict[str, List[float]]    # 2ª entrada (vel. angular / vel. linear)
    rules: List[List[str]]              # matriz 3×3 — linhas: var2, colunas: var1
    output_mfs: Dict[str, List[float]]  # FPs triangulares de saída [a, b, c]
    labels: List[str] = field(default_factory=lambda: ['N', 'Z', 'P'])


@dataclass
class CartPoleParams:
    """Parâmetros físicos do sistema pêndulo-carro (pág. 7)."""
    mc: float = 0.5    # massa do carro [kg]
    mp: float = 0.2    # massa do pêndulo [kg]
    l:  float = 0.3    # comprimento da haste [m]
    g:  float = 9.8    # aceleração gravitacional [m/s²]
    I:  float = 0.006  # momento de inércia [kg·m²]


# ---------------------------------------------------------------------------
# Funções de pertinência
# ---------------------------------------------------------------------------

def trapmf(x: float, params: List[float]) -> float:
    """
    Função de pertinência trapezoidal.
    params = [a, b, c, d]
    Trapézios degenerados (N e P dos extremos) são suportados:
    quando a == b ou c == d, a rampa correspondente retorna 1.
    """
    a, b, c, d = params
    left  = (x - a) / (b - a) if b > a else 1.0
    right = (d - x) / (d - c) if d > c else 1.0
    return float(np.clip(min(left, right), 0.0, 1.0))


def fuzzify(x: float, mfs: Dict[str, List[float]]) -> Dict[str, float]:
    """Retorna o grau de pertinência de x em cada conjunto fuzzy."""
    return {label: trapmf(x, params) for label, params in mfs.items()}


# ---------------------------------------------------------------------------
# Motor de inferência Mamdani + defuzzificação Weighted Average
# ---------------------------------------------------------------------------

def infer_and_defuzz(
    mu1: Dict[str, float],
    mu2: Dict[str, float],
    rules: List[List[str]],
    output_mfs: Dict[str, List[float]],
    labels: List[str],
) -> float:
    """
    Inferência Mamdani (t-norma = mínimo) + Weighted Average (pág. 7):

        z* = Σ μ(z̄) · z̄  /  Σ μ(z̄)

    z̄ é o vértice central B de cada FP triangular consequente.
    """
    weighted_sum = 0.0
    weight_total = 0.0

    for i, lv2 in enumerate(labels):       # var2 → linhas da matriz de regras
        for j, lv1 in enumerate(labels):   # var1 → colunas da matriz de regras
            firing = min(mu2[lv2], mu1[lv1])
            if firing <= 0.0:
                continue
            consequent = rules[i][j]
            center = output_mfs[consequent][1]   # vértice B da triangular
            weighted_sum += firing * center
            weight_total += firing

    return weighted_sum / weight_total if weight_total > 0.0 else 0.0


def compute_control_force(system: FuzzySystem, val1: float, val2: float) -> float:
    """
    Interface de alto nível: fuzzifica → infere → defuzzifica.

    Convenção:
      val1 → var1 (colunas da regra: ângulo  ou posição)
      val2 → var2 (linhas  da regra: vel.ang. ou vel.lin.)
    """
    mu1 = fuzzify(val1, system.var1_mfs)
    mu2 = fuzzify(val2, system.var2_mfs)
    return infer_and_defuzz(mu1, mu2, system.rules, system.output_mfs, system.labels)


# ---------------------------------------------------------------------------
# Dinâmica do sistema — equações acopladas sequenciais (pág. 7)
# ---------------------------------------------------------------------------

def equations_of_motion(
    state: np.ndarray,
    F: float,
    p: CartPoleParams,
) -> Tuple[float, float]:
    """
    Equações do pêndulo-carro conforme o PDF (pág. 7):

        ẍ = [ mp·l·(θ̇²·sin θ − θ̈·cos θ) + F ] / (mc + mp)   … (1)
        θ̈ = mp·l·[g·sin θ − ẍ·cos θ]            / (I + mp·l²) … (2)

    Resolução sequencial: θ̈ é desconhecido em (1), portanto o termo
    (−θ̈·cos θ) é desconsiderado no cálculo de ẍ. O ẍ resultante é
    então substituído em (2) para obter θ̈. Essa é a formulação
    explicitamente apresentada nas equações da pág. 7.
    """
    _, _, theta, theta_dot = state
    sin_t = np.sin(theta)
    cos_t = np.cos(theta)

    # (1) Aceleração linear — sem o termo θ̈
    x_acc = (p.mp * p.l * theta_dot**2 * sin_t + F) / (p.mc + p.mp)

    # (2) Aceleração angular — usa ẍ calculado acima
    theta_acc = (p.mp * p.l * (p.g * sin_t - x_acc * cos_t)) / (p.I + p.mp * p.l**2)

    return x_acc, theta_acc


def step_euler_explicit(
    state: np.ndarray,
    F: float,
    p: CartPoleParams,
    h: float,
) -> np.ndarray:
    """
    Integração por Euler Explícito conforme o loop da pág. 8:

        ẋ_new  = ẋ_old  + h · ẍ          (vel. linear nova)
        x_new  = x_old  + h · ẋ_old      (posição com vel. ANTIGA)
        θ̇_new = θ̇_old + h · θ̈          (vel. angular nova)
        θ_new  = θ_old  + h · θ̇_old     (ângulo com vel. ANTIGA)
    """
    x_acc, theta_acc = equations_of_motion(state, F, p)

    new_state = state.copy()
    new_state[1] = state[1] + h * x_acc        # ẋ nova
    new_state[0] = state[0] + h * state[1]     # x com ẋ antiga
    new_state[3] = state[3] + h * theta_acc    # θ̇ nova
    new_state[2] = state[2] + h * state[3]     # θ com θ̇ antiga

    return new_state


# ---------------------------------------------------------------------------
# Definição dos controladores fuzzy (parâmetros extraídos do PDF)
# ---------------------------------------------------------------------------

def build_pendulum_controller() -> FuzzySystem:
    """
    Controlador do Pêndulo (págs. 2–4).

    FPs do ângulo theta (pág. 3, tabela "vértices trapezoidais"):
      N: [-0.15, -0.15, -0.10,  0.00]   (rampa desce de -0.1 a 0)
      Z: [-0.10, -0.03,  0.03,  0.10]   (triangular simétrico)
      P: [ 0.00,  0.10,  0.15,  0.15]   (rampa sobe de 0 a 0.1)

    FPs da vel. angular theta_dot (pág. 3, tabela inferior):
      N: [-0.15, -0.15, -0.10,  0.00]
      Z: [-0.15, -0.03,  0.03,  0.15]   (base mais larga que o ângulo)
      P: [ 0.00,  0.10,  0.15,  0.15]

    FPs de saída — triangulares (pág. 4):
      NL[-200,-100,0]  NM[-80,-40,0]  NS[-10,-5,0]  Z[0,0,0]
      PS[0,5,10]  PM[0,40,80]  PL[0,100,200]

    Matriz de regras (pág. 2) — linhas: θ̇, colunas: θ:
      θ̇╲θ |  N    Z    P
        N  | NL   NS   Z
        Z  | NM   Z    PM
        P  | Z    PS   PL
    """
    theta_mfs = {
        'N': [-0.15, -0.15, -0.10,  0.00],
        'Z': [-0.10, -0.03,  0.03,  0.10],
        'P': [ 0.00,  0.10,  0.15,  0.15],
    }
    omega_mfs = {
        'N': [-0.15, -0.15, -0.10,  0.00],
        'Z': [-0.15, -0.03,  0.03,  0.15],
        'P': [ 0.00,  0.10,  0.15,  0.15],
    }
    rules = [
        ['NL', 'NS', 'Z' ],   # θ̇ = N
        ['NM', 'Z',  'PM'],   # θ̇ = Z
        ['Z',  'PS', 'PL'],   # θ̇ = P
    ]
    output_mfs = {
        'NL': [-200, -100,   0],
        'NM': [ -80,  -40,   0],
        'NS': [ -10,   -5,   0],
        'Z':  [   0,    0,   0],
        'PS': [   0,    5,  10],
        'PM': [   0,   40,  80],
        'PL': [   0,  100, 200],
    }
    return FuzzySystem(theta_mfs, omega_mfs, rules, output_mfs)


def build_cart_controller() -> FuzzySystem:
    """
    Controlador do Carro (págs. 5–6).

    FPs da posição x (pág. 5, tabela + gráfico):
      N: [-3, -3, -2,  0]   (tabela: C=-2, D=0; platô à esquerda)
      Z: [-1.5, -0.5, 0.5, 1.5]   (tabela: A=-1.5, B=-0.5, C=0.5, D=1.5)
      P: [ 0,  2,  3,  3]   (tabela: A=0, B=2; platô à direita)

    FPs da velocidade x_dot (pág. 6, tabela + gráfico — simetria em relação à posição):
      N: [-3, -3, -2,  0]
      Z: [-1.5, -0.5, 0.5, 1.5]
      P: [ 0,  2,  3,  3]

    FPs de saída — triangulares (pág. 6):
      NL[-100,-50,0]  NM[-10,-5,0]  NS[-2,-1,0]  Z[0,0,0]
      PS[0,1,2]  PM[0,5,10]  PL[0,50,100]

    Matriz de regras (pág. 5) — linhas: ẋ, colunas: x:
      ẋ╲x |  N    Z    P
        N  | PL   PS   Z
        Z  | PM   Z    NM
        P  | Z    NS   NL
    """
    pos_mfs = {
        'N': [-3.0, -3.0, -2.0,  0.0],
        'Z': [-1.5, -0.5,  0.5,  1.5],
        'P': [ 0.0,  2.0,  3.0,  3.0],
    }
    vel_mfs = {
        'N': [-3.0, -3.0, -2.0,  0.0],
        'Z': [-1.5, -0.5,  0.5,  1.5],
        'P': [ 0.0,  2.0,  3.0,  3.0],
    }
    rules = [
        ['PL', 'PS', 'Z' ],   # ẋ = N
        ['PM', 'Z',  'NM'],   # ẋ = Z
        ['Z',  'NS', 'NL'],   # ẋ = P
    ]
    output_mfs = {
        'NL': [-100,  -50,   0],
        'NM': [ -10,   -5,   0],
        'NS': [  -2,   -1,   0],
        'Z':  [   0,    0,   0],
        'PS': [   0,    1,   2],
        'PM': [   0,    5,  10],
        'PL': [   0,   50, 100],
    }
    return FuzzySystem(pos_mfs, vel_mfs, rules, output_mfs)


# ---------------------------------------------------------------------------
# Limites de segurança
# ---------------------------------------------------------------------------

THETA_LIMIT = np.pi / 2   # pêndulo caiu se |θ| > 90°
X_LIMIT     = 2.4         # carro saiu dos trilhos se |x| > 2.4 m


def is_terminal(state: np.ndarray) -> bool:
    """Retorna True se o sistema atingiu condição de falha."""
    return bool(abs(state[2]) > THETA_LIMIT or abs(state[0]) > X_LIMIT)


# ---------------------------------------------------------------------------
# Simulação principal
# ---------------------------------------------------------------------------

def simulate(
    initial_state: List[float],
    n_steps: int = 500,
    h: float = 0.02,
    print_every: int = 10,
) -> List[np.ndarray]:
    """
    Executa a simulação do pêndulo invertido com controle fuzzy.

    Parâmetros
    ----------
    initial_state : [x, x_dot, theta, theta_dot]
    n_steps       : número de passos de integração
    h             : passo de tempo [s]  (sugestão do PDF: 0.02 s)
    print_every   : intervalo de impressão (0 = silencioso)

    Retorna
    -------
    Lista de arrays [x, x_dot, theta, theta_dot] a cada passo.
    """
    pendulum_ctrl = build_pendulum_controller()
    cart_ctrl     = build_cart_controller()
    params        = CartPoleParams()

    state = np.array(initial_state, dtype=float)
    trajectory: List[np.ndarray] = [state.copy()]

    if print_every > 0:
        header = (
            f"{'Tempo':>6} | {'x (m)':>8} | {'x_dot':>8} | "
            f"{'theta':>8} | {'th_dot':>8} | {'F (N)':>8}"
        )
        print(header)
        print("─" * len(header))

    for step in range(n_steps):
        x, x_dot, theta, theta_dot = state

        # ── Controle fuzzy ─────────────────────────────────────────────────
        # Pêndulo: var1 = theta (colunas), var2 = theta_dot (linhas)
        f_pendulo = compute_control_force(pendulum_ctrl, theta, theta_dot)

        # Carro: var1 = x (colunas), var2 = x_dot (linhas)
        f_carro   = compute_control_force(cart_ctrl, x, x_dot)

        F = f_pendulo + f_carro

        # ── Integração Euler Explícito (pág. 8) ───────────────────────────
        state = step_euler_explicit(state, F, params, h)
        trajectory.append(state.copy())

        if print_every > 0 and step % print_every == 0:
            t = step * h
            print(
                f"{t:6.2f}s | {state[0]:8.4f} | {state[1]:8.4f} | "
                f"{state[2]:8.4f} | {state[3]:8.4f} | {F:8.2f}"
            )

        if is_terminal(state):
            if print_every > 0:
                print(f"\n[!] Divergência no passo {step} (t = {step * h:.2f} s)")
                print(f"    |theta| = {abs(state[2]):.4f} rad  |x| = {abs(state[0]):.4f} m")
            break

    return trajectory


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    initial = [0.0, 0.0, 0.1, 0.0]   # pêndulo a 0.1 rad (~5.7°)

    print("=" * 72)
    print("  Pêndulo Invertido sobre Carro — Controlador Fuzzy FIS")
    print(f"  Estado inicial: x={initial[0]} m | x_dot={initial[1]} m/s | "
          f"theta={initial[2]} rad | th_dot={initial[3]} rad/s")
    print("=" * 72)
    print()

    trajectory = simulate(initial, n_steps=500, h=0.02, print_every=10)

    final = trajectory[-1]
    print(f"\n{'─' * 72}")
    print("Estado final:")
    print(f"  x      = {final[0]:+.6f} m")
    print(f"  x_dot  = {final[1]:+.6f} m/s")
    print(f"  theta  = {final[2]:+.6f} rad  ({np.degrees(final[2]):+.4f}°)")
    print(f"  th_dot = {final[3]:+.6f} rad/s")
    print(f"  Passos executados: {len(trajectory) - 1}")


  Pêndulo Invertido sobre Carro — Controlador Fuzzy FIS
  Estado inicial: x=0.0 m | x_dot=0.0 m/s | theta=0.1 rad | th_dot=0.0 rad/s

 Tempo |    x (m) |    x_dot |    theta |   th_dot |    F (N)
─────────────────────────────────────────────────────────────
  0.00s |   0.0000 |   1.1429 |   0.1000 |  -2.7940 |    40.00
  0.20s |   0.0549 |   0.7504 |  -0.0204 |  -1.7573 |    -5.43
  0.40s |   0.0781 |  -0.2886 |  -0.0538 |   0.8774 |   -12.19
  0.60s |   0.0587 |   0.0631 |   0.0146 |  -0.0626 |    -0.27
  0.80s |   0.0725 |   0.0345 |   0.0042 |   0.0498 |     4.71
  1.00s |   0.0859 |  -0.0477 |  -0.0007 |   0.2691 |    -5.24
  1.20s |   0.0978 |  -0.0075 |   0.0004 |   0.1818 |    -6.94
  1.40s |   0.1112 |  -0.0812 |   0.0009 |   0.3804 |    -7.41
  1.60s |   0.1267 |   0.0314 |  -0.0007 |   0.1167 |    -7.43
  1.80s |   0.1390 |   0.2902 |   0.0087 |  -0.5146 |     7.47
  2.00s |   0.1564 |  -0.0545 |   0.0085 |   0.3664 |    -7.91
  2.20s |   0.1770 |  -0.0574 |   0.0062 |   0.40